# 00 - Relevamiento de los dumps de Lichess

**Objetivo:** medir cuantas partidas del dump mensual sobreviven al filtro
(ELO >= 2200 a ambos jugadores, controles Blitz / Rapid / Classical) **antes** de
gastar horas de computo en el etiquetado con Stockfish.

Este paso es barato: recorre el dump por streaming y solo cuenta. No descarga el
archivo entero ni invoca al motor. Con el numero que devuelve se decide si
alcanza con un solo mes o si hay que sumar dumps.

> **Runtime:** usar **CPU**, no GPU. Este notebook no entrena nada.

Corresponde a la tarea 3.1 del WBS (descarga y exploracion de partidas).

## 1. Entorno

In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# Stockfish con version fija (queda registrada en cada fila del dataset).
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
from chessdl.colab import describe_runtime

runtime = describe_runtime("stockfish")
print(runtime.summary())

## 2. Configuracion

Todos los parametros del pipeline salen de `configs/dataset_v1.yaml`, que es la
unica fuente de verdad y se versiona junto con el codigo (requerimiento 2.3).

In [ ]:
from chessdl.config import load_config

cfg = load_config()
print("Dumps configurados :", cfg.source.dumps)
print("ELO minimo         :", cfg.filter.min_elo, "(exigido a ambos jugadores)")
print("Controles de tiempo:", cfg.filter.time_controls)
print("Posiciones/partida :", cfg.sampling.positions_per_game)

## 3. Relevamiento

`--max-scanned` limita cuantas partidas se **miran**. Un millon de partidas
alcanza para estimar la tasa de aceptacion con buena precision y tarda pocos
minutos; sin el limite recorre el dump completo.

Nota: la tasa estimada sobre el principio del archivo puede sesgarse levemente,
porque el dump viene ordenado por fecha dentro del mes.

In [ ]:
from chessdl.data import pipeline

resultados = pipeline.survey(cfg, max_scanned=1_000_000)

for stats in resultados:
    print(stats.summary())

### La misma operación desde la línea de comando

Las celdas de arriba usan la API de Python. El repositorio expone además un
comando equivalente, que es el que documenta el README como vía de reproducción
(requerimiento 2.2). Desde una celda se invoca con `!`:

In [ ]:
!{sys.executable} -m chessdl.scripts.build_dataset --survey --max-scanned 200000 --quiet

## 4. Cuantos meses hacen falta

Extrapolamos de la tasa medida al dump completo. Un dump mensual de Lichess
ronda las 90-100 millones de partidas.

In [ ]:
PARTIDAS_POR_DUMP = 95_000_000   # orden de magnitud de un mes de Lichess
OBJETIVO_POSICIONES = 2_000_000  # objetivo para entrenar la ResNet

tasa = sum(s.games_accepted for s in resultados) / max(sum(s.games_seen for s in resultados), 1)
partidas_por_mes = PARTIDAS_POR_DUMP * tasa
posiciones_por_mes = partidas_por_mes * cfg.sampling.positions_per_game

print(f"Tasa de aceptacion medida : {tasa:.4%}")
print(f"Partidas utiles por mes   : {partidas_por_mes:,.0f}")
print(f"Posiciones por mes        : {posiciones_por_mes:,.0f}")
print()
meses = max(1, round(OBJETIVO_POSICIONES / max(posiciones_por_mes, 1) + 0.49))
print(f"Para {OBJETIVO_POSICIONES:,} posiciones hacen falta ~{meses} mes(es) de dump.")

Si hace falta mas de un mes, agregar los meses adicionales a `source.dumps`
en `configs/dataset_v1.yaml`. El pipeline los procesa en orden y guarda el
progreso por `(dump, offset)`, asi que se pueden sumar meses despues sin
rehacer nada de lo ya etiquetado.

**Proximo paso:** `01_build_dataset.ipynb`.